# GMSI Hourly Index Live Polling

## Integration Guide: Global Macro Sentiment Indices — Hourly Index

This notebook demonstrates how to integrate the **Permutable AI GMSI Hourly Index API** into an analytical workflow. The GMSI index pre-aggregates raw headlines into one-hour windows, keyed by **country**, **topic**, **sentiment type**, and **index type**.

### Output schema

| Field | Description |
|---|---|
| `publication_time` | End of the hourly aggregation window (UTC) |
| `headline_count` | Headlines contributing to the window |
| `sentiment_sum` | Sum of scores for the selected sentiment type |
| `sentiment_abs_sum` | Sum of absolute scores — total signal magnitude |
| `sentiment_std` | Standard deviation — disagreement within the window |

**Derived metrics:**

```python
sentiment_avg     = sentiment_sum / headline_count        # mean per-headline, ∈ [−1, +1]
activity_weighted = sentiment_sum / sentiment_abs_sum     # volume-adjusted direction
```

**Two sentiment types are available:**

| Sentiment type | Meaning |
|---|---|
| `directional` | Topic-specific direction (e.g. +1 = inflation rising) |
| `semantic_sentiment` | General language tone (+1 = upbeat, −1 = alarming) |

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing here constitutes financial advice or a recommendation to buy, sell, or hold any asset.

### Endpoint

| Endpoint | Path |
|---|---|
| Live | `GET /v1/headlines/index/gmsi/live/{country}` |
| Historical | `GET /v1/headlines/index/gmsi/historical/{country}` |

Key parameters: `topic`, `sentiment_type` (`directional` or `semantic_sentiment`), `index_type` (`COMBINED` / `DOMESTIC` / `INTERNATIONAL`), `limit`.

**Important:** The API omits `country`, `topic`, `sentiment_type`, and `index_type` from each JSON record — stamp them from request parameters when storing.

### Pipeline

1. Install dependencies and configure credentials
2. Define fetch and upsert helpers
3. Optional historical backfill
4. Dry-run a single poll
5. Visualise sentiment_avg and activity_weighted
6. Run the 15-minute polling loop
7. Visualise the accumulated window
8. Compare directional vs semantic divergence
9. Derive a rolling sentiment indicator

In [ ]:
%pip install requests pandas matplotlib seaborn plotly


## Step 1 — Configuration

In [ ]:
import getpass
import os
from pathlib import Path


def load_env_file() -> None:
    candidates = []
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.append(base / '.env')
        candidates.append(base / 'systematic/global_macro_sentiment/app/live_index_polling/.env')

    for env_path in candidates:
        if not env_path.exists():
            continue
        with env_path.open() as env_file:
            for line in env_file:
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    key, value = line.split('=', 1)
                    os.environ.setdefault(key.strip(), value.strip().strip('"'))
        return


load_env_file()
API_KEY = os.environ.get('API_KEY') or os.environ.get('PERMUTABLE_API_KEY')
if not API_KEY:
    try:
        API_KEY = getpass.getpass('Enter your Permutable AI API key: ')
    except Exception as exc:
        raise RuntimeError('Set API_KEY in permutable-examples/.env or export API_KEY before running this notebook.') from exc

BASE_URL = 'https://copilot-api.permutable.ai/v1'
HEADERS = {'x-api-key': API_KEY}

# ── Poll configuration ────────────────────────────────────────────────────────
COUNTRIES       = ['US', 'JP', 'IT', 'GB', 'FR', 'DE', 'CA']                # ISO-2 codes
TOPICS          = ['Monetary Policy-Policy Tools-Interest Rates', 'Economic Data-Inflation-Energy', 'Economic Data-Inflation-Food', 'Fiscal Policy-Fiscal Stance']  # full topic names
INDEX_TYPES     = ['COMBINED']                             # DOMESTIC | INTERNATIONAL | COMBINED
SENTIMENT_TYPES = ['directional', 'semantic_sentiment']    # both models
POLL_INTERVAL_SECONDS = 900
BACKFILL_DAYS = 7
DB_PATH = '/tmp/gmsi_index_nb.db'

n_combos = len(COUNTRIES) * len(TOPICS) * len(INDEX_TYPES) * len(SENTIMENT_TYPES)
print(f'Config: {n_combos} combinations per poll cycle  ({len(COUNTRIES)}c × {len(TOPICS)}t × {len(INDEX_TYPES)}it × {len(SENTIMENT_TYPES)}st)')




## Step 2 — Fetch and storage helpers

In [ ]:
import sqlite3
import requests
import pandas as pd
from datetime import datetime, timezone, timedelta, date

# ── SQLite schema ─────────────────────────────────────────────────────────────

def setup_database(db_path: str = DB_PATH) -> None:
    with sqlite3.connect(db_path) as conn:
        conn.execute('''
            CREATE TABLE IF NOT EXISTS gmsi_index (
                country           TEXT NOT NULL,
                topic             TEXT NOT NULL,
                sentiment_type    TEXT NOT NULL,
                index_type        TEXT NOT NULL,
                publication_time  TEXT NOT NULL,
                headline_count    INTEGER,
                sentiment_sum     REAL,
                sentiment_abs_sum REAL,
                sentiment_std     REAL,
                fetched_at        TEXT,
                PRIMARY KEY (country, topic, sentiment_type, index_type, publication_time)
            )
        ''')
        conn.commit()

setup_database()
print('Database ready:', DB_PATH)

# ── Upsert ────────────────────────────────────────────────────────────────────

INDEX_COLUMNS = [
    'country', 'topic', 'sentiment_type', 'index_type', 'publication_time',
    'headline_count', 'sentiment_sum', 'sentiment_abs_sum', 'sentiment_std', 'fetched_at',
]

def upsert_index(records: list, db_path: str = DB_PATH) -> int:
    if not records:
        return 0
    ph = ', '.join(['?'] * len(INDEX_COLUMNS))
    sql = f"INSERT OR REPLACE INTO gmsi_index ({', '.join(INDEX_COLUMNS)}) VALUES ({ph})"
    rows = [tuple(r.get(c) for c in INDEX_COLUMNS) for r in records]
    with sqlite3.connect(db_path) as conn:
        conn.executemany(sql, rows)
        conn.commit()
    return len(rows)

# ── Fetch helpers ─────────────────────────────────────────────────────────────

def _stamp(records: list, country: str, topic: str, sentiment_type: str, index_type: str) -> None:
    ts = datetime.now(timezone.utc).isoformat()
    for r in records:
        r['country']        = country
        r['topic']          = topic
        r['sentiment_type'] = sentiment_type
        r['index_type']     = index_type
        r['fetched_at']     = ts

def fetch_live_index(country: str, topic: str, sentiment_type: str,
                     index_type: str) -> list:
    url = f'{BASE_URL}/headlines/index/gmsi/live/{country}'
    r = requests.get(url, headers=HEADERS, params={
        'topic': topic, 'sentiment_type': sentiment_type,
        'index_type': index_type,
        'sparse': 'true', 'align_to_period_end': 'true',
        'limit': 1000,
    }, timeout=30)
    r.raise_for_status()
    records = r.json()
    _stamp(records, country, topic, sentiment_type, index_type)
    return records

def fetch_historical_index(country: str, topic: str, sentiment_type: str,
                            index_type: str, start_date: date) -> list:
    url = f'{BASE_URL}/headlines/index/gmsi/historical/{country}'
    params = {
        'topic': topic, 'sentiment_type': sentiment_type,
        'index_type': index_type,
        'sparse': 'true', 'align_to_period_end': 'true',
        'start_date': start_date.isoformat(), 'limit': 1000,
    }
    all_records = []
    while True:
        r = requests.get(url, headers=HEADERS, params=params, timeout=30)
        r.raise_for_status()
        body = r.json()
        _stamp(body['data'], country, topic, sentiment_type, index_type)
        all_records.extend(body['data'])
        if not body['has_more']:
            break
        params['next_token'] = body['next_token']
    return all_records

print('Helpers defined.')


## Step 3 — Historical backfill (optional)

In [ ]:
RUN_BACKFILL = False   # Set to True to run the historical backfill

if RUN_BACKFILL:
    start = (datetime.utcnow() - timedelta(days=BACKFILL_DAYS)).date()
    for country in COUNTRIES:
        for topic in TOPICS:
            for sentiment_type in SENTIMENT_TYPES:
                for index_type in INDEX_TYPES:
                    key = f'{country}/{topic}/{sentiment_type}/{index_type}'
                    print(f'  {key}...')
                    try:
                        records = fetch_historical_index(country, topic, sentiment_type, index_type, start)
                        n = upsert_index(records)
                        print(f'    {n} rows upserted')
                    except Exception as exc:
                        print(f'    ERROR: {exc}')
    print('Backfill complete.')
else:
    print('Backfill skipped — set RUN_BACKFILL = True for populated charts.')


## Step 4 — Dry-run: single poll

In [ ]:
# Fetch directional index for the first configured country/topic
sample = fetch_live_index(COUNTRIES[0], TOPICS[0], 'directional', INDEX_TYPES[0])
n = upsert_index(sample)
print(f'Fetched {len(sample)} records for {COUNTRIES[0]} / {TOPICS[0]} / directional / {INDEX_TYPES[0]} | {n} upserted')

if sample:
    df_sample = pd.DataFrame(sample)
    df_sample['publication_time'] = pd.to_datetime(df_sample['publication_time'], utc=True)
    df_sample['sentiment_avg'] = (
        df_sample['sentiment_sum'] / df_sample['headline_count'].replace(0, float('nan'))
    ).fillna(0.0)
    df_sample['activity_weighted'] = (
        df_sample['sentiment_sum'] / df_sample['sentiment_abs_sum'].replace(0, float('nan'))
    ).fillna(0.0)
    display(df_sample[[
        'country', 'topic', 'sentiment_type', 'index_type', 'publication_time',
        'headline_count', 'sentiment_avg', 'activity_weighted', 'sentiment_std',
    ]].head(10))
else:
    print('No records returned — the topic may have no recent hourly buckets for this country.')


## Step 5 — Visualise the initial snapshot

Plot `sentiment_avg` and `activity_weighted` for the initial data.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def load_db(db_path: str = DB_PATH, hours: int = 48,
             sentiment_type: str = 'directional',
             index_type: str = 'COMBINED') -> pd.DataFrame:
    since = (datetime.now(timezone.utc) - timedelta(hours=hours)).isoformat()
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql(
            '''SELECT * FROM gmsi_index
               WHERE publication_time >= ?
                 AND sentiment_type = ?
                 AND index_type = ?
               ORDER BY publication_time''',
            conn, params=(since, sentiment_type, index_type),
        )
    df['publication_time'] = pd.to_datetime(df['publication_time'], utc=True)
    df['sentiment_avg'] = (
        df['sentiment_sum'] / df['headline_count'].replace(0, float('nan'))
    ).fillna(0.0)
    df['activity_weighted'] = (
        df['sentiment_sum'] / df['sentiment_abs_sum'].replace(0, float('nan'))
    ).fillna(0.0)
    return df

df = load_db(hours=48, sentiment_type='directional')
print(f'{len(df)} directional records loaded')

if df.empty:
    print('No data yet — run the dry-run cell above or the backfill step.')
else:
    countries = df['country'].unique()
    fig, axes = plt.subplots(len(countries), 1, figsize=(14, 3.5 * len(countries)), sharex=True)
    if len(countries) == 1:
        axes = [axes]

    ROLLING_W = 5
    for ax, country in zip(axes, countries):
        sub = df[df['country'] == country].sort_values('publication_time')
        avg_smooth = sub['sentiment_avg'].rolling(ROLLING_W, min_periods=1).mean()
        aw_smooth  = sub['activity_weighted'].rolling(ROLLING_W, min_periods=1).mean()
        ax.plot(sub['publication_time'], sub['sentiment_avg'],
                color='#3498db', lw=1, alpha=0.4, linestyle='--', label='sentiment_avg (raw)')
        ax.plot(sub['publication_time'], avg_smooth,
                color='#3498db', lw=2, label=f'sentiment_avg ({ROLLING_W}h rolling)')
        ax.plot(sub['publication_time'], aw_smooth,
                color='#e67e22', lw=2, label='activity_weighted (rolling)', linestyle='-.')
        ax.axhline(0, color='grey', lw=0.8, linestyle=':')
        ax.set_ylim(-1.1, 1.1)
        ax.set_ylabel(country, rotation=0, labelpad=28, fontsize=11)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
        if ax is axes[0]:
            ax.legend(loc='upper right', fontsize=9)
            ax.set_title(f'GMSI Directional Index  |  {TOPICS[0]}  |  COMBINED  |  {ROLLING_W}h rolling')

    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## Step 6 — Live polling loop

In [ ]:
import time

RUN_POLLING_LOOP = False   # Set to True to start the live polling loop

if RUN_POLLING_LOOP:
    poll = 0
    while True:
        poll += 1
        t0 = datetime.now(timezone.utc)
        print(f'[Poll {poll}]  {t0.strftime("%H:%M:%S UTC")}')
        for country in COUNTRIES:
            for topic in TOPICS:
                for sentiment_type in SENTIMENT_TYPES:
                    for index_type in INDEX_TYPES:
                        key = f'{country}/{topic[:20]}/{sentiment_type}/{index_type}'
                        try:
                            records = fetch_live_index(country, topic, sentiment_type, index_type)
                            n = upsert_index(records)
                            print(f'  {key}: {len(records)} fetched | {n} upserted')
                        except Exception as exc:
                            print(f'  {key}: ERROR — {exc}')
        elapsed = (datetime.now(timezone.utc) - t0).total_seconds()
        wait = max(0, POLL_INTERVAL_SECONDS - elapsed)
        print(f'  Done in {elapsed:.1f}s.  Next poll in {wait/60:.1f} min.\n')
        time.sleep(wait)
else:
    print('Polling loop disabled — set RUN_POLLING_LOOP = True to enable.')


## Step 7 — Directional vs semantic divergence over time

Plot both sentiment types for the same country on one chart. Divergences often precede regime shifts.

In [ ]:
# Load both sentiment types from DB
dir_df = load_db(hours=168, sentiment_type='directional')
sem_df = load_db(hours=168, sentiment_type='semantic_sentiment')

if dir_df.empty and sem_df.empty:
    print('No data yet.')
else:
    ROLLING_W = 5
    countries = sorted(set(dir_df['country'].unique()) | set(sem_df['country'].unique()))
    fig, axes = plt.subplots(len(countries), 1, figsize=(14, 3.5 * len(countries)), sharex=True)
    if len(countries) == 1:
        axes = [axes]

    for ax, country in zip(axes, countries):
        for source_df, label, color, ls in [
            (dir_df, 'directional', '#3498db', '-'),
            (sem_df, 'semantic', '#e67e22', '--'),
        ]:
            sub = source_df[source_df['country'] == country].sort_values('publication_time')
            if sub.empty:
                continue
            smooth = sub.set_index('publication_time')['sentiment_avg'].rolling(ROLLING_W, min_periods=1).mean()
            ax.plot(smooth.index, smooth.values, color=color, lw=2, linestyle=ls, label=label)

        ax.axhline(0, color='grey', lw=0.8, linestyle=':')
        ax.set_ylim(-1.1, 1.1)
        ax.set_ylabel(country, rotation=0, labelpad=28, fontsize=11)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
        if ax is axes[0]:
            ax.legend(loc='upper right', fontsize=9)
            ax.set_title(f'Directional vs Semantic  |  {TOPICS[0]}  |  {ROLLING_W}h rolling')

    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


## Step 8 — Headline volume heatmap

Visualise news activity across countries and time.

In [ ]:
import seaborn as sns

if not dir_df.empty:
    pivot_count = dir_df.pivot_table(
        index='country',
        columns=dir_df['publication_time'].dt.floor('6h').rename('hour'),
        values='headline_count',
        aggfunc='sum',
    )
    fig, ax = plt.subplots(figsize=(14, max(3, len(pivot_count) * 0.7)))
    sns.heatmap(
        pivot_count, cmap='Blues', linewidths=0.4, ax=ax,
        cbar_kws={'label': 'Headline count'},
    )
    ax.set_title(f'Headline Count per 6h Window  |  {TOPICS[0]}  |  COMBINED')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('No data yet.')


## Step 9 — Rolling sentiment indicator

Derive a HIGH / NEUTRAL / LOW signal and print the latest indicator per country.

In [ ]:
UPPER_THRESHOLD = 0.5
LOWER_THRESHOLD = -0.5
ROLLING_WINDOW  = 5   # hours

if not dir_df.empty:
    hourly = (
        dir_df
        .sort_values(['country', 'publication_time'])
        .groupby(['country', 'publication_time'])
        .agg(
            sentiment_sum=('sentiment_sum', 'sum'),
            sentiment_abs_sum=('sentiment_abs_sum', 'sum'),
            headline_count=('headline_count', 'sum'),
        )
        .reset_index()
    )
    hourly['sentiment_avg'] = (
        hourly['sentiment_sum'] / hourly['headline_count'].replace(0, float('nan'))
    ).fillna(0.0)
    hourly['activity_weighted'] = (
        hourly['sentiment_sum'] / hourly['sentiment_abs_sum'].replace(0, float('nan'))
    ).fillna(0.0)
    hourly['sentiment_smooth'] = hourly.groupby('country')['sentiment_avg'].transform(
        lambda x: x.rolling(ROLLING_WINDOW, min_periods=1).mean()
    )
    hourly['indicator'] = 'NEUTRAL'
    hourly.loc[hourly['sentiment_smooth'] >= UPPER_THRESHOLD, 'indicator'] = 'HIGH'
    hourly.loc[hourly['sentiment_smooth'] <= LOWER_THRESHOLD, 'indicator'] = 'LOW'

    latest = (
        hourly.sort_values('publication_time')
        .groupby('country').last().reset_index()
        [['country', 'publication_time', 'sentiment_smooth', 'activity_weighted', 'headline_count', 'indicator']]
        .sort_values('sentiment_smooth', ascending=False)
    )

    print(f'Latest directional indicator  |  {TOPICS[0]}  |  COMBINED')
    print(f'Rolling window: {ROLLING_WINDOW}h  |  Thresholds: {LOWER_THRESHOLD} / {UPPER_THRESHOLD}')
    display(latest)

    # Rolling indicator heatmap
    pivot_smooth = hourly.pivot_table(
        index='country', columns='publication_time',
        values='sentiment_smooth', aggfunc='mean',
    )
    fig, ax = plt.subplots(figsize=(14, max(3, len(pivot_smooth) * 0.7)))
    sns.heatmap(
        pivot_smooth, cmap='RdYlGn', vmin=-1, vmax=1, center=0,
        linewidths=0.4, ax=ax, cbar_kws={'label': 'Directional smooth'},
    )
    ax.set_title(f'{ROLLING_WINDOW}h Rolling Directional Avg  |  {TOPICS[0]}')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('No directional data — run the dry-run or backfill cells first.')
